# Customer Segmentation Project — K-Means Clustering
### Exploratory Data Analysis, Customer Spending Patterns & Clustering

This notebook follows the workflow visible in the supplied tutorial screenshots and uses the Kaggle **Customer Segmentation: Clustering** dataset.

**Before running:** download `customer_segmentation.csv` from the provided Kaggle dataset page and place it in the same folder as this notebook.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 2. Load the dataset

In [ ]:
df = pd.read_csv("customer_segmentation.csv")
df.head()

## 3. Inspect the dataset

In [ ]:
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.describe(include="all").T

## 4. Data cleaning and feature engineering

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Convert the customer registration date to datetime when available
if "Dt_Customer" in df.columns:
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True, errors="coerce")

# Create interpretable customer features from the source columns
if "Year_Birth" in df.columns:
    df["Age"] = 2025 - pd.to_numeric(df["Year_Birth"], errors="coerce")

spend_cols = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
spend_cols = [col for col in spend_cols if col in df.columns]
if spend_cols:
    df["Total_Spending"] = df[spend_cols].sum(axis=1)

child_cols = [col for col in ["Kidhome", "Teenhome"] if col in df.columns]
if child_cols:
    df["Total_Children"] = df[child_cols].sum(axis=1)

print("Shape after removing duplicates:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
display(df.head())

## 5. Exploratory data analysis (EDA)

In [ ]:
if "Income" in df.columns:
    sns.histplot(df["Income"].dropna(), bins=30, kde=True)
    plt.title("Income Distribution")
    plt.xlabel("Income")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
if "Total_Spending" in df.columns:
    sns.histplot(df["Total_Spending"], bins=30, kde=True)
    plt.title("Total Spending Distribution")
    plt.xlabel("Total Spending")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
if "Age" in df.columns:
    sns.histplot(df["Age"].dropna(), bins=30, kde=True)
    plt.title("Customer Age Distribution")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Spending by product category
if spend_cols:
    spending_by_category = df[spend_cols].sum().sort_values(ascending=False)
    ax = spending_by_category.plot(kind="bar", color="steelblue", edgecolor="black")
    ax.set_title("Total Spending by Product Category")
    ax.set_xlabel("Product category")
    ax.set_ylabel("Total spend")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Correlation analysis

In [ ]:
corr_cols = [c for c in [
    "Income", "Age", "Recency", "Total_Spending",
    "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases",
    "NumWebVisitsMonth", "Total_Children"
] if c in df.columns]

if len(corr_cols) >= 2:
    corr = df[corr_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=.5)
    plt.title("Correlation Heatmap of Customer Features")
    plt.tight_layout()
    plt.show()

## 7. Income comparison by education and marital status

In [ ]:
if all(c in df.columns for c in ["Income", "Education", "Marital_Status"]):
    pivot_income = df.pivot_table(
        values="Income",
        index="Education",
        columns="Marital_Status",
        aggfunc="mean"
    )
    display(pivot_income.round(2))
    plt.figure(figsize=(10, 5))
    sns.heatmap(pivot_income, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title("Average Income by Education and Marital Status")
    plt.tight_layout()
    plt.show()

## 8. Prepare features for clustering

K-Means is distance-based, so selected numeric features are standardized. Identifier columns and campaign-response flags are excluded from the default model features. Missing values are imputed using the median for numeric features.

In [ ]:
candidate_features = [
    "Income", "Age", "Recency", "Total_Spending",
    "NumDealsPurchases", "NumWebPurchases",
    "NumCatalogPurchases", "NumStorePurchases",
    "NumWebVisitsMonth", "Total_Children"
]
features = [c for c in candidate_features if c in df.columns]

if len(features) < 2:
    raise ValueError(
        "At least two expected numeric features were not found. "
        "Check that you downloaded the tutorial's customer_segmentation.csv dataset."
    )

X = df[features].apply(pd.to_numeric, errors="coerce")
# Remove rows with no usable values in the selected features, then median-impute
valid_rows = X.notna().any(axis=1)
X = X.loc[valid_rows].copy()
df_model = df.loc[valid_rows].copy()
X = X.fillna(X.median(numeric_only=True))

# Avoid constant columns, which add no information to clustering
features = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
X = X[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Rows used:", len(X))
print("Features used:", features)

## 9. Select the number of clusters — Elbow Method

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

## 10. Compare silhouette scores

In [ ]:
silhouette_scores = []
for k in k_values:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

plt.plot(list(k_values), silhouette_scores, marker="o", color="darkorange")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores
}).round(3))

## 11. Fit K-Means and assign customer segments

In [ ]:
# Choose K after reviewing the elbow curve and silhouette scores.
# The default is 4; change it if your diagnostics and interpretation support another value.
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_model["Cluster"] = kmeans.fit_predict(X_scaled)
df_model["Segment"] = df_model["Cluster"].map(lambda x: f"Segment {x + 1}")

print("Silhouette score for selected K:",
      round(silhouette_score(X_scaled, df_model["Cluster"]), 3))
display(df_model[["Cluster", "Segment"] + features].head())

## 12. Visualize the segments

In [ ]:
# Spending vs income plot (when both features are available)
x_feature = "Income" if "Income" in features else features[0]
y_feature = "Total_Spending" if "Total_Spending" in features else features[1]

plot_data = df_model.copy()
sns.scatterplot(
    data=plot_data, x=x_feature, y=y_feature,
    hue="Segment", palette="Set2", alpha=.8
)
plt.title(f"Customer Segments: {x_feature} vs {y_feature}")
plt.tight_layout()
plt.show()

In [ ]:
cluster_counts = df_model["Segment"].value_counts().sort_index()
ax = cluster_counts.plot(kind="bar", color="cornflowerblue", edgecolor="black")
ax.set_title("Number of Customers in Each Segment")
ax.set_xlabel("Segment")
ax.set_ylabel("Customer count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 13. Profile the customer segments

In [ ]:
profile = df_model.groupby(["Cluster", "Segment"])[features].mean().round(2)
profile["Customers"] = df_model.groupby("Cluster").size()
profile["Customer_Share_Percent"] = (
    profile["Customers"] / len(df_model) * 100
).round(2)
display(profile)

In [ ]:
# Compare segment averages for the selected behavioral/demographic features
profile_features = [c for c in features if c in profile.columns]
plt.figure(figsize=(12, 6))
sns.heatmap(profile[profile_features], annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Average Customer Characteristics by Segment")
plt.tight_layout()
plt.show()

## 14. Business insights and interpretation

In [ ]:
overall_means = df_model[features].mean()
for cluster_id, row in profile.iterrows():
    print(f"\\n{cluster_id[1]} — {int(row['Customers'])} customers "
          f"({row['Customer_Share_Percent']:.1f}%)")
    differences = []
    for feature in features:
        if feature not in row or pd.isna(row[feature]) or overall_means[feature] == 0:
            continue
        ratio = row[feature] / overall_means[feature]
        if ratio >= 1.15:
            differences.append(f"above-average {feature}")
        elif ratio <= 0.85:
            differences.append(f"below-average {feature}")
    print(", ".join(differences[:5]) if differences else
          "close to the overall average across the selected features")

## 15. Export the final results

In [ ]:
output = df_model.copy()
output.to_csv("customers_with_segments.csv", index=False)
profile.reset_index().to_csv("customer_segment_profile.csv", index=False)
print("Saved customers_with_segments.csv")
print("Saved customer_segment_profile.csv")

## Conclusion

This notebook explores customer demographics, purchase behavior, and product-category spending before applying K-Means clustering. The elbow method and silhouette score help compare candidate cluster counts; segment profiles then support interpretation and marketing hypotheses.

**Important:** segment names are neutral until their averages have been reviewed. The data represents customers in the supplied dataset, not necessarily all customers in a real business. Validate proposed marketing actions against business context and campaign results.